# 🔍 DeepInspect AI — Complete Self-Contained Google Colab Notebook
### Unsupervised Industrial Anomaly Detection using PyTorch Convolutional Autoencoder

هذا الملف متكامل ومكتفٍ ذاتياً (100% Self-Contained). يمكنك تشغيله في **Google Colab** مباشرة بضغطة زر واحدة (**Runtime ➔ Run all**) دون الحاجة لرفع أي ملفات أخرى!

---
### 📌 خريطة عمل النوت بوك:
1. **فحص تسريع كرت الشاشة (NVIDIA CUDA GPU)**
2. **تجهيز وبناء كافة ملفات وأكواد المشروع برمجياً** (`configs/`, `src/`)
3. **تثبيت المتطلبات البرمجية**
4. **تحميل بيانات الزجاجات MVTec AD Bottle تلقائياً** (بدون الحاجة لأي حساب أو توكن)
5. **تدريب النموذج عبر كرت الشاشة** وحفظ أفضل أوزان في `models/best_model.pth`
6. **عرض منحنى خسارة التدريب (Loss Curve)**
7. **تقييم الأداء وحساب مقاييس الدقة** (Image AUROC, Pixel AUROC, F1-Score)
8. **عرض بصري تفاعلي للخرائط الحرارية (Heatmaps & Defect Localization)**
9. **تحميل النموذج المدرب لجهازك المحلي بضغطة زر** لتشغيله في لوحة Streamlit

## الخطوة 1: فحص كرت الشاشة (GPU Acceleration)
تأكد من أن نوع بيئة التشغيل هو **T4 GPU** (`Runtime` ➔ `Change runtime type` ➔ `T4 GPU`).

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")

## الخطوة 2: إنشاء كافة مجلدات وأكواد المشروع برمجياً (Self-Contained Code Generation)
تقوم هذه الخلية ببناء هيكلية المشروع ومجلدات `src/` و `configs/` تلقائياً داخل Colab.

In [ ]:
import os
from pathlib import Path

# 1. إنشاء المجلدات
folders = [
    "configs", "models", "outputs", "data",
    "src/data", "src/models", "src/training", "src/evaluation", "src/inference"
]
for f in folders:
    os.makedirs(f, exist_ok=True)

# 2. إنشاء configs/config.yaml
with open("configs/config.yaml", "w", encoding="utf-8") as f:
    f.write('''project:
  name: "DeepInspect-AI"
  seed: 42
dataset:
  name: "MVTec-AD"
  category: "bottle"
  root_dir: "data/mvtec_ad/bottle"
  image_size: [256, 256]
  channels: 3
  batch_size: 16
  num_workers: 2
  val_split_ratio: 0.2
  pin_memory: true
model:
  name: "ConvAutoencoder"
  in_channels: 3
  channel_multipliers: [32, 64, 128, 256]
  latent_dim: 128
training:
  epochs: 50
  learning_rate: 0.001
  weight_decay: 0.00001
  device: "cuda"
  checkpoint_dir: "./models"
  output_dir: "./outputs"
''')

# 3. إنشاء src/data/dataset.py
with open("src/data/dataset.py", "w", encoding="utf-8") as f:
    f.write('''from pathlib import Path
import numpy as np
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
import yaml

class MVTecDataset(Dataset):
    def __init__(self, root_dir, split="train", image_size=(256, 256), val_ratio=0.2, seed=42):
        self.root_dir = Path(root_dir).expanduser()
        self.split = split
        self.image_size = tuple(image_size)
        self.val_ratio = float(val_ratio)
        self.seed = int(seed)
        self.transform = T.Compose([
            T.Resize(self.image_size, interpolation=InterpolationMode.BILINEAR),
            T.ToTensor(),
        ])
        self.target_transform = T.Compose([
            T.Resize(self.image_size, interpolation=InterpolationMode.NEAREST),
            T.ToTensor(),
        ])
        self.samples = []
        self._load_samples()

    def _load_samples(self):
        train_dir = self.root_dir / "train" / "good"
        test_dir = self.root_dir / "test"
        gt_dir = self.root_dir / "ground_truth"

        if self.split in ("train", "val"):
            if not train_dir.exists(): return
            images = sorted([p for p in train_dir.iterdir() if p.suffix.lower() in (".png", ".jpg", ".jpeg")])
            rng = np.random.RandomState(self.seed)
            indices = np.arange(len(images))
            rng.shuffle(indices)
            val_count = int(np.floor(len(images) * self.val_ratio))
            sel = indices[:val_count] if self.split == "val" else indices[val_count:]
            for idx in sel:
                self.samples.append({"image_path": images[idx], "label": 0, "mask_path": None})
        elif self.split == "test":
            if not test_dir.exists(): return
            for defect in sorted([d.name for d in test_dir.iterdir() if d.is_dir()]):
                is_anom = (defect != "good")
                for img_path in sorted((test_dir / defect).glob("*.png")):
                    mask_path = gt_dir / defect / f"{img_path.stem}_mask.png"
                    self.samples.append({
                        "image_path": img_path,
                        "label": 1 if is_anom else 0,
                        "mask_path": mask_path if mask_path.exists() else None
                    })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        meta = self.samples[idx]
        with Image.open(meta["image_path"]) as img:
            image = self.transform(img.convert("RGB"))
        if meta["mask_path"] and meta["mask_path"].exists():
            with Image.open(meta["mask_path"]) as m:
                mask = (self.target_transform(m.convert("L")) > 0.5).float()
        else:
            mask = torch.zeros((1, *self.image_size), dtype=torch.float32)
        return image, meta["label"], mask, str(meta["image_path"])

def get_dataloaders(config):
    if isinstance(config, (str, Path)):
        with open(config, "r") as f: config = yaml.safe_load(f)
    ds = config.get("dataset", {})
    root = ds.get("root_dir", "data/mvtec_ad/bottle")
    size = tuple(ds.get("image_size", [256, 256]))
    bs = int(ds.get("batch_size", 16))
    val_r = float(ds.get("val_split_ratio", 0.2))
    
    train_ds = MVTecDataset(root, "train", size, val_r)
    val_ds = MVTecDataset(root, "val", size, val_r)
    test_ds = MVTecDataset(root, "test", size, val_r)
    
    return (
        DataLoader(train_ds, batch_size=bs, shuffle=True, pin_memory=True),
        DataLoader(val_ds, batch_size=bs, shuffle=False),
        DataLoader(test_ds, batch_size=bs, shuffle=False)
    )
''')

# 4. إنشاء src/models/autoencoder.py
with open("src/models/autoencoder.py", "w", encoding="utf-8") as f:
    f.write('''import torch
import torch.nn as nn
import yaml

class Encoder(nn.Module):
    def __init__(self, in_channels=3, channels=None, latent_dim=128, input_size=(256, 256)):
        super().__init__()
        channels = channels or [32, 64, 128, 256]
        blocks = []
        prev = in_channels
        for ch in channels:
            blocks.extend([
                nn.Conv2d(prev, ch, 3, 1, 1, bias=False),
                nn.BatchNorm2d(ch),
                nn.ReLU(True),
                nn.MaxPool2d(2, 2)
            ])
            prev = ch
        self.features = nn.Sequential(*blocks)
        with torch.no_grad():
            dummy = self.features(torch.zeros(1, in_channels, *input_size))
            self.bottleneck_shape = dummy.shape[1:]
            self.flat_dim = dummy.numel()
        self.fc = nn.Linear(self.flat_dim, latent_dim)

    def forward(self, x):
        return self.fc(self.features(x).flatten(1))

class Decoder(nn.Module):
    def __init__(self, latent_dim=128, out_channels=3, channels=None, bottleneck_shape=(256, 16, 16)):
        super().__init__()
        channels = channels or [256, 128, 64, 32]
        self.bottleneck_shape = bottleneck_shape
        self.fc = nn.Linear(latent_dim, int(torch.tensor(bottleneck_shape).prod().item()))
        deconvs = []
        for i in range(len(channels) - 1):
            deconvs.extend([
                nn.ConvTranspose2d(channels[i], channels[i+1], 2, 2, bias=False),
                nn.BatchNorm2d(channels[i+1]),
                nn.ReLU(True)
            ])
        deconvs.extend([
            nn.ConvTranspose2d(channels[-1], out_channels, 2, 2),
            nn.Sigmoid()
        ])
        self.deconv = nn.Sequential(*deconvs)

    def forward(self, z):
        return self.deconv(self.fc(z).view(-1, *self.bottleneck_shape))

class ConvAutoencoder(nn.Module):
    def __init__(self, in_channels=3, input_size=(256, 256), channels=None, latent_dim=128):
        super().__init__()
        channels = channels or [32, 64, 128, 256]
        self.encoder = Encoder(in_channels, channels, latent_dim, input_size)
        self.decoder = Decoder(latent_dim, in_channels, list(reversed(channels)), self.encoder.bottleneck_shape)

    def forward(self, x):
        return self.decoder(self.encoder(x))

def build_autoencoder(config):
    if isinstance(config, str):
        with open(config) as f: config = yaml.safe_load(f)
    m = config.get("model", {})
    d = config.get("dataset", {})
    return ConvAutoencoder(
        in_channels=m.get("in_channels", 3),
        input_size=tuple(d.get("image_size", [256, 256])),
        channels=list(m.get("channel_multipliers", [32, 64, 128, 256])),
        latent_dim=int(m.get("latent_dim", 128))
    )
''')

# 5. إنشاء src/training/train.py
with open("src/training/train.py", "w", encoding="utf-8") as f:
    f.write('''import argparse, sys, yaml
from pathlib import Path
import torch
import torch.nn as nn
from torch.optim import Adam
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path(__file__).resolve().parent.parent.parent))
from src.data.dataset import get_dataloaders
from src.models.autoencoder import build_autoencoder

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/config.yaml")
    parser.add_argument("--data_dir", default="data/mvtec_ad/bottle")
    parser.add_argument("--checkpoint_dir", default="./models")
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--device", default="cuda")
    args = parser.parse_args()

    with open(args.config) as f: cfg = yaml.safe_load(f)
    cfg["dataset"]["root_dir"] = args.data_dir
    cfg["dataset"]["batch_size"] = args.batch_size
    
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "cuda" else "cpu")
    print(f"==================================================")
    print(f" DeepInspect AI - Training on {device} ({torch.cuda.get_device_name(0) if device.type=='cuda' else 'CPU'})")
    print(f" Total Epochs: {args.epochs} | Batch Size: {args.batch_size}")
    print(f"==================================================")

    train_loader, val_loader, _ = get_dataloaders(cfg)
    model = build_autoencoder(cfg).to(device)
    optimizer = Adam(model.parameters(), lr=args.lr, weight_decay=1e-5)
    criterion = nn.MSELoss()

    best_loss = float("inf")
    Path(args.checkpoint_dir).mkdir(parents=True, exist_ok=True)
    Path("outputs").mkdir(parents=True, exist_ok=True)
    history = {"train": [], "val": []}

    for epoch in range(1, args.epochs + 1):
        model.train()
        train_loss = 0.0
        for imgs, _, _, _ in train_loader:
            imgs = imgs.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, imgs)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
        train_loss /= max(len(train_loader.dataset), 1)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, _, _, _ in val_loader:
                imgs = imgs.to(device)
                val_loss += criterion(model(imgs), imgs).item() * imgs.size(0)
        val_loss /= max(len(val_loader.dataset), 1)

        history["train"].append(train_loss)
        history["val"].append(val_loss)

        saved = ""
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "config": cfg}, f"{args.checkpoint_dir}/best_model.pth")
            saved = " -> [SAVED BEST]"

        print(f"[Epoch {epoch:02d}/{args.epochs:02d}] Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}{saved}")

    plt.figure(figsize=(8, 4))
    plt.plot(history["train"], label="Train MSE")
    plt.plot(history["val"], label="Val MSE")
    plt.legend()
    plt.title("DeepInspect AI - Reconstruction Loss")
    plt.savefig("outputs/loss_curve.png")
    print(f"\\n✅ Training complete! Best model saved to {args.checkpoint_dir}/best_model.pth")

if __name__ == "__main__":
    main()
''')

# 6. إنشاء src/evaluation/metrics.py
with open("src/evaluation/metrics.py", "w", encoding="utf-8") as f:
    f.write('''import argparse, sys, yaml, cv2, json
from pathlib import Path
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path(__file__).resolve().parent.parent.parent))
from src.data.dataset import get_dataloaders
from src.models.autoencoder import build_autoencoder

def evaluate():
    with open("configs/config.yaml") as f: cfg = yaml.safe_load(f)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    _, _, test_loader = get_dataloaders(cfg)
    model = build_autoencoder(cfg).to(device)
    ckpt = torch.load("models/best_model.pth", map_location=device)
    model.load_state_dict(ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt)
    model.eval()

    y_true_img, y_scores_img = [], []
    y_true_pix, y_scores_pix = [], []

    print("Evaluating test dataset...")
    with torch.no_grad():
        for imgs, labels, masks, _ in test_loader:
            imgs = imgs.to(device)
            recon = model(imgs)
            res = torch.mean((imgs - recon) ** 2, dim=1).cpu().numpy()
            masks_np = masks.cpu().numpy()

            for b in range(imgs.size(0)):
                smoothed = cv2.GaussianBlur(res[b], (7, 7), 4.0)
                y_scores_img.append(float(np.max(smoothed)))
                y_true_img.append(int(labels[b]))
                y_true_pix.extend(masks_np[b, 0].flatten())
                y_scores_pix.extend(smoothed.flatten())

    img_auroc = roc_auc_score(y_true_img, y_scores_img)
    pix_auroc = roc_auc_score(y_true_pix, y_scores_pix)
    
    prec, rec, thresh = precision_recall_curve(y_true_img, y_scores_img)
    f1 = 2 * (prec * rec) / (prec + rec + 1e-8)
    opt_thresh = float(thresh[np.argmax(f1)])

    print("=" * 60)
    print(f"📊 Image-Level AUROC : {img_auroc:.4f}")
    print(f"📊 Pixel-Level AUROC : {pix_auroc:.4f}")
    print(f"🎯 Optimal Threshold : {opt_thresh:.6f}")
    print("=" * 60)

    results = {"image_auroc": img_auroc, "pixel_auroc": pix_auroc, "optimal_threshold": opt_thresh}
    with open("outputs/evaluation_results.json", "w") as f: json.dump(results, f, indent=4)

if __name__ == "__main__":
    evaluate()
''')

# 7. إنشاء ملفات __init__.py
for p in ["src", "src/data", "src/models", "src/training", "src/evaluation", "src/inference"]:
    with open(f"{p}/__init__.py", "w") as f: f.write("")

print("✅ تم بناء جميع ملفات المشروع بنجاح داخل بيئة Colab!")

## الخطوة 3: تثبيت المكتبات اللازمة

In [ ]:
!pip install -q pyyaml opencv-python matplotlib tqdm scikit-learn pillow huggingface_hub

## الخطوة 4: تحميل بيانات الزجاجات MVTec AD Bottle
يتم التحميل مباشرة من مستودع Hugging Face المعتمد والمفتوح (`foersben/mvtec-ad`) بدون أي توكن وبأقصى سرعة.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

# إزالة المجلد السابق لضمان تنظيف أي روابط رمزية مكسورة (Symlinks)
!rm -rf data/mvtec_ad

print("⬇️ جاري تحميل بيانات الزجاجات من Hugging Face...")
downloaded_path = snapshot_download(
    repo_id="foersben/mvtec-ad",
    repo_type="dataset",
    allow_patterns="bottle/*",
)

print("📂 جاري نسخ وتنظيم المجلدات وفك الروابط الرمزية (-L)...")
!mkdir -p data/mvtec_ad
!cp -r -L {downloaded_path}/bottle data/mvtec_ad/

# التحقق من أن الملفات حقيقية وليست روابط معطوبة
test_sample = "data/mvtec_ad/bottle/train/good/000.png"
if os.path.exists(test_sample):
    size_kb = os.path.getsize(test_sample) / 1024
    print(f"✅ تم تجهيز واستخراج البيانات بنجاح! حجم الصورة: {size_kb:.1f} KB")
else:
    print("⚠️ تحذير: لم يتم العثور على الصور.")

!ls -lh data/mvtec_ad/bottle/train/good | head -n 5


## الخطوة 5: تدريب الـ Convolutional Autoencoder على الـ GPU

In [ ]:
!python src/training/train.py \
    --config configs/config.yaml \
    --data_dir data/mvtec_ad/bottle \
    --checkpoint_dir ./models \
    --epochs 50 \
    --batch_size 16 \
    --lr 0.001 \
    --device cuda

### عرض منحنى الخسارة (Loss Curve)

In [ ]:
from IPython.display import Image, display
import os

if os.path.exists("outputs/loss_curve.png"):
    display(Image("outputs/loss_curve.png", width=700))
else:
    print("لم يتم العثور على منحنى الخسارة. تأكد من اكتمال التدريب.")

## الخطوة 6: تقييم النموذج وحساب مقاييس الدقة (AUROC & Optimal F1)

In [ ]:
!python src/evaluation/metrics.py

## الخطوة 7: فحص بصري تفاعلي وتوليد الخرائط الحرارية للعيوب داخل Colab

In [ ]:
import matplotlib.pyplot as plt
import cv2, torch, yaml
import numpy as np
from PIL import Image
from pathlib import Path
from src.models.autoencoder import build_autoencoder

with open("configs/config.yaml") as f: cfg = yaml.safe_load(f)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_autoencoder(cfg).to(device)
ckpt = torch.load("models/best_model.pth", map_location=device)
model.load_state_dict(ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt)
model.eval()

test_dir = Path("data/mvtec_ad/bottle/test")
sample_types = ["good", "broken_large", "broken_small", "contamination"]
fig, axes = plt.subplots(len(sample_types), 4, figsize=(16, 4 * len(sample_types)))

for i, stype in enumerate(sample_types):
    img_files = sorted(list((test_dir / stype).glob("*.png")))
    if not img_files: continue
    
    pil_img = Image.open(img_files[0]).convert("RGB").resize((256, 256))
    orig_np = np.array(pil_img)
    tensor_img = (torch.from_numpy(orig_np).permute(2, 0, 1).float() / 255.0).unsqueeze(0).to(device)
    
    with torch.no_grad():
        recon_tensor = model(tensor_img)
        res = torch.mean((tensor_img - recon_tensor)**2, dim=1).squeeze().cpu().numpy()
        recon_np = (recon_tensor.squeeze().permute(1, 2, 0).cpu().numpy().clip(0, 1) * 255).astype(np.uint8)

    smoothed = cv2.GaussianBlur(res, (7, 7), 4.0)
    score = np.max(smoothed)
    norm = ((smoothed - smoothed.min()) / (smoothed.max() - smoothed.min() + 1e-8) * 255).astype(np.uint8)
    heatmap = cv2.applyColorMap(norm, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(orig_np, 0.6, cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB), 0.4, 0)
    
    verdict = "PASS (NORMAL)" if stype == "good" else "REJECT (DEFECT)"
    v_color = "green" if stype == "good" else "red"

    axes[i, 0].imshow(orig_np); axes[i, 0].set_title(f"[{stype.upper()}] Input"); axes[i, 0].axis("off")
    axes[i, 1].imshow(recon_np); axes[i, 1].set_title("Autoencoder Reconstruction"); axes[i, 1].axis("off")
    axes[i, 2].imshow(heatmap); axes[i, 2].set_title(f"Heatmap (Score: {score:.4f})"); axes[i, 2].axis("off")
    axes[i, 3].imshow(overlay); axes[i, 3].set_title(f"Verdict: {verdict}", color=v_color, fontweight="bold"); axes[i, 3].axis("off")

plt.suptitle("DeepInspect AI - Defect Localization Breakdown", fontsize=16, fontweight="bold", y=1.002)
plt.tight_layout()
plt.show()


## الخطوة 8: تحميل النموذج لجهازك المحلي (Download Checkpoint)
شغّل هذه الخلية لتحميل ملف `best_model.pth` مباشرة إلى حاسوبك لتشغيله في لوحة Streamlit.

In [ ]:
from google.colab import files
import os

if os.path.exists("models/best_model.pth"):
    print("⬇️ جاري تحميل best_model.pth إلى جهازك...")
    files.download("models/best_model.pth")
else:
    print("ملف best_model.pth غير موجود. تأكد من إتمام التدريب أولاً!")